# Lab 01: Numerical Derivatives & Activation Function Gradients

[Back to Session Overview](../README.md) · [Module 02 Index](../../README.md)

---

## Overview

In machine learning and deep learning, model training relies on calculating gradients of objective loss functions with respect to network parameters. This process requires evaluating derivatives of non-linear **activation functions** chained together across network layers.

In this notebook, we explore the foundational mechanics of derivatives:
1. **The Difference Quotient:** Computing instantaneous rates of change via limits.
2. **Numerical Differentiation Schemes:** Comparing single-sided forward/backward differences against symmetric central differences.
3. **Truncation Error vs. Floating-Point Cancellation:** Analyzing the trade-offs of finite step sizes $h$.
4. **Analytical Derivatives of Six Core Activations:** Deriving and implementing Sigmoid, Exponential, PReLU, ELU, Softplus, and Softsign.
5. **Numerical Gradient Checking:** Building a reusable diagnostic tool to verify analytical derivatives against central difference approximations.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

# Set plotting style for clean visualization
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---

## 1. Mathematical Foundation: Numerical Differentiation

The analytical derivative of a continuous function $f(x)$ at point $x$ is defined as the limit of the difference quotient:

$$f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}$$

On a digital computer with IEEE 754 floating-point arithmetic, we cannot set $h \to 0$. Instead, we choose a small finite perturbation $h > 0$:

1. **Forward Difference (One-Sided Right):**
   $$D_+ f(x) = \frac{f(x + h) - f(x)}{h} = f'(x) + \mathcal{O}(h)$$

2. **Backward Difference (One-Sided Left):**
   $$D_- f(x) = \frac{f(x) - f(x - h)}{h} = f'(x) + \mathcal{O}(h)$$

3. **Central Difference (Symmetric):**
   $$D_0 f(x) = \frac{f(x + h) - f(x - h)}{2h} = f'(x) + \mathcal{O}(h^2)$$

Through Taylor series expansions, the symmetric central difference cancels odd-order derivative terms, achieving quadratic convergence $\mathcal{O}(h^2)$ compared to linear $\mathcal{O}(h)$ for one-sided schemes.


In [ ]:
def forward_difference(f, x, h=1e-5):
    """Compute forward (one-sided right) numerical difference."""
    return (f(x + h) - f(x)) / h


def backward_difference(f, x, h=1e-5):
    """Compute backward (one-sided left) numerical difference."""
    return (f(x) - f(x - h)) / h


def central_difference(f, x, h=1e-5):
    """Compute symmetric central numerical difference (second-order accurate)."""
    return (f(x + h) - f(x - h)) / (2.0 * h)


# Benchmark on f(x) = x^2 + 2x at x = 3.0
# Analytical f'(x) = 2x + 2 -> f'(3.0) = 8.0
def test_f(x):
    return x**2 + 2.0 * x


analytical_val = 8.0
x_val = 3.0

fwd_val = forward_difference(test_f, x_val, h=1e-3)
bwd_val = backward_difference(test_f, x_val, h=1e-3)
cen_val = central_difference(test_f, x_val, h=1e-3)

print(f"Target analytical f'(3.0): {analytical_val:.6f}")
print(f"Forward  Difference (h=1e-3): {fwd_val:.6f} (Error: {abs(fwd_val - analytical_val):.2e})")
print(f"Backward Difference (h=1e-3): {bwd_val:.6f} (Error: {abs(bwd_val - analytical_val):.2e})")
print(f"Central  Difference (h=1e-3): {cen_val:.6f} (Error: {abs(cen_val - analytical_val):.2e})")

assert abs(cen_val - analytical_val) < 1e-10, "Central difference should be exact for quadratic!"

---

## 2. Six Common Activation Functions & Analytical Derivatives

Activation functions introduce non-linear mappings required for deep architectures to learn complex representations. Here we derive and implement each activation and its exact derivative.

### 2.1 Sigmoid Function

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

$$\sigma'(x) = \frac{d}{dx}(1 + e^{-x})^{-1} = -(1 + e^{-x})^{-2} (-e^{-x}) = \frac{e^{-x}}{(1 + e^{-x})^2} = \sigma(x)(1 - \sigma(x))$$


In [ ]:
def sigmoid(x):
    """Numerically stable sigmoid function."""
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-x)), np.exp(x) / (1.0 + np.exp(x)))


def sigmoid_derivative(x):
    """Analytical derivative: sigma(x) * (1 - sigma(x))."""
    s = sigmoid(x)
    return s * (1.0 - s)


# Verify at x = 0: sigmoid(0) = 0.5, sigmoid'(0) = 0.25
assert np.isclose(sigmoid(0.0), 0.5)
assert np.isclose(sigmoid_derivative(0.0), 0.25)

### 2.2 Exponential Function

$$f(x) = e^x, \qquad f'(x) = e^x$$


In [ ]:
def exponential(x):
    """Exponential activation f(x) = exp(x)."""
    return np.exp(x)


def exponential_derivative(x):
    """Analytical derivative f'(x) = exp(x)."""
    return np.exp(x)


assert np.isclose(exponential(1.0), math.e)
assert np.isclose(exponential_derivative(1.0), math.e)

### 2.3 Parametric ReLU (PReLU)

$$\text{PReLU}(x) = \begin{cases} \alpha x & \text{if } x < 0 \\ x & \text{if } x \ge 0 \end{cases}, \qquad \text{PReLU}'(x) = \begin{cases} \alpha & \text{if } x \le 0 \\ 1 & \text{if } x > 0 \end{cases}$$

Here we adopt $\alpha = 0.1$ as highlighted in the lecture slides.


In [ ]:
def prelu(x, alpha=0.1):
    """Parametric ReLU with slope alpha for x < 0."""
    return np.where(x >= 0, x, alpha * x)


def prelu_derivative(x, alpha=0.1):
    """Analytical derivative of PReLU."""
    return np.where(x >= 0, 1.0, alpha)


assert np.isclose(prelu(-2.0, alpha=0.1), -0.2)
assert np.isclose(prelu(2.0, alpha=0.1), 2.0)
assert np.isclose(prelu_derivative(-2.0, alpha=0.1), 0.1)
assert np.isclose(prelu_derivative(2.0, alpha=0.1), 1.0)

### 2.4 Exponential Linear Unit (ELU)

$$\text{ELU}(x) = \begin{cases} \alpha(e^x - 1) & \text{if } x \le 0 \\ x & \text{if } x > 0 \end{cases}, \qquad \text{ELU}'(x) = \begin{cases} \alpha e^x & \text{if } x \le 0 \\ 1 & \text{if } x > 0 \end{cases}$$

With $\alpha = 0.1$, the function has continuous value and first derivative at $x = 0$ (since $\alpha e^0 = \alpha = 0.1$ when matching left limit).


In [ ]:
def elu(x, alpha=0.1):
    """Exponential Linear Unit."""
    return np.where(x > 0, x, alpha * (np.exp(x) - 1.0))


def elu_derivative(x, alpha=0.1):
    """Analytical derivative of ELU."""
    return np.where(x > 0, 1.0, alpha * np.exp(x))


assert np.isclose(elu(0.0, alpha=0.1), 0.0)
assert np.isclose(elu(2.0, alpha=0.1), 2.0)
assert np.isclose(elu_derivative(2.0, alpha=0.1), 1.0)

### 2.5 Softplus Function

$$\text{softplus}(x) = \log(1 + e^x)$$

$$\text{softplus}'(x) = \frac{e^x}{1 + e^x} = \frac{1}{1 + e^{-x}} = \sigma(x)$$

Remarkably, the derivative of Softplus is exactly the Sigmoid function!


In [ ]:
def softplus(x):
    """Smooth approximation to ReLU: log(1 + exp(x))."""
    # Numerically stable formulation
    return np.where(x > 20.0, x, np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0))


def softplus_derivative(x):
    """Analytical derivative equals sigmoid(x)."""
    return sigmoid(x)


# Verify derivative identity at multiple test points
xs = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
assert np.allclose(softplus_derivative(xs), sigmoid(xs))

### 2.6 Softsign Function

$$\text{softsign}(x) = \frac{x}{1 + |x|}$$

Derivation:
- For $x > 0$: $f(x) = \frac{x}{1 + x} \implies f'(x) = \frac{(1+x)(1) - x(1)}{(1+x)^2} = \frac{1}{(1+x)^2}$
- For $x < 0$: $f(x) = \frac{x}{1 - x} \implies f'(x) = \frac{(1-x)(1) - x(-1)}{(1-x)^2} = \frac{1}{(1-x)^2}$

Unifying across both domains:
$$\text{softsign}'(x) = \frac{1}{(1 + |x|)^2}$$


In [ ]:
def softsign(x):
    """Softsign activation: x / (1 + |x|)."""
    return x / (1.0 + np.abs(x))


def softsign_derivative(x):
    """Analytical derivative: 1 / (1 + |x|)^2."""
    return 1.0 / ((1.0 + np.abs(x)) ** 2)


assert np.isclose(softsign(0.0), 0.0)
assert np.isclose(softsign_derivative(0.0), 1.0)

---

## 3. Automated Gradient Checking

Gradient checking computes the relative difference between the analytical gradient and the numerical central difference approximation:

$$\text{Relative Error} = \frac{|f'_{\text{analytical}}(x) - f'_{\text{numerical}}(x)|}{\max(|f'_{\text{analytical}}(x)|, |f'_{\text{numerical}}(x)|) + \epsilon}$$

A relative error below $10^{-5}$ confirms that the analytical derivative implementation is free from algebraic or coding defects.


In [ ]:
def gradient_check(f, f_prime, x, h=1e-5, eps=1e-8):
    """Verify analytical derivative against central differences."""
    num_grad = central_difference(f, x, h=h)
    ana_grad = f_prime(x)
    rel_error = abs(ana_grad - num_grad) / (max(abs(ana_grad), abs(num_grad)) + eps)
    return rel_error


activations = [
    ("Sigmoid", sigmoid, sigmoid_derivative),
    ("Exponential", exponential, exponential_derivative),
    ("PReLU (alpha=0.1)", lambda x: prelu(x, alpha=0.1), lambda x: prelu_derivative(x, alpha=0.1)),
    ("ELU (alpha=0.1)", lambda x: elu(x, alpha=0.1), lambda x: elu_derivative(x, alpha=0.1)),
    ("Softplus", softplus, softplus_derivative),
    ("Softsign", softsign, softsign_derivative),
]

test_points = [-3.0, -1.5, -0.5, 0.5, 1.5, 3.0]
print(
    f"{'Activation':<22} | {'Point':<6} | {'Analytical':<12} | {'Numerical':<12} | {'Rel Error':<10} | {'Status'}"
)
print("-" * 80)

for name, f_act, f_prime in activations:
    for pt in test_points:
        ana = float(f_prime(pt))
        num = float(central_difference(f_act, pt))
        err = gradient_check(f_act, f_prime, pt)
        status = "PASS" if err < 1e-4 else "FAIL"
        print(f"{name:<22} | {pt:>6.2f} | {ana:>12.6f} | {num:>12.6f} | {err:>10.2e} | {status}")
        assert err < 1e-4, f"Gradient check failed for {name} at {pt}!"

---

## 4. Activation Function & Derivative Visualizations

Here we visualize the six activation functions alongside their exact derivative curves across $x \in [-5, 5]$.


In [ ]:
x_grid = np.linspace(-5.0, 5.0, 500)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

funcs = [
    ("Sigmoid", sigmoid, sigmoid_derivative),
    ("Exponential (truncated y)", exponential, exponential_derivative),
    ("PReLU (alpha=0.1)", lambda x: prelu(x, 0.1), lambda x: prelu_derivative(x, 0.1)),
    ("ELU (alpha=0.1)", lambda x: elu(x, 0.1), lambda x: elu_derivative(x, 0.1)),
    ("Softplus", softplus, softplus_derivative),
    ("Softsign", softsign, softsign_derivative),
]

for idx, (title, fn, fn_p) in enumerate(funcs):
    ax = axes[idx]
    y_val = fn(x_grid)
    y_grad = fn_p(x_grid)

    ax.plot(x_grid, y_val, label="f(x)", color="royalblue", lw=2)
    ax.plot(x_grid, y_grad, label="f '(x)", color="crimson", lw=2, linestyle="--")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("x")
    ax.set_ylabel("Output")
    ax.legend(loc="upper left")
    if "Exponential" in title:
        ax.set_ylim(-0.5, 20.0)

plt.tight_layout()
plt.show()